# Node 3 — Content Validation playground

Demonstrates `ContentValidationNode` end-to-end:
OCR routing → min-chars check → language detection (lingua) → SLM legitimacy gate → SSE events → audit record.

Node 3 always runs **after** Node 2: it receives the extracted `text: str` plus the
`FileReceptionResult` from Node 1 (needed for OCR routing on image-only PDFs).

> **Kernel**: select `.venv` (Python 3.12) in the top-right kernel picker.

## 1 — Imports

In [1]:
import asyncio

from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

from classiflow.ingesta.domain.results import FileReceptionResult
from classiflow.ingesta.nodes.node3_content_validation import ContentValidationNode
from classiflow.shared.audit.service import AuditService
from classiflow.shared.database.base import Base
from classiflow.shared.database.repositories.audit import SqlAuditRepository
from classiflow.shared.events.broadcaster import EventBroadcaster

print("imports OK")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


## 2 — Database setup

Creates the SQLite engine and ensures all tables exist.
The `session_factory` is reused by every section below.

In [2]:
from pathlib import Path

import classiflow.settings as _settings_mod

# settings.py lives at src/classiflow/settings.py → parents[2] = project root
_project_root = Path(_settings_mod.__file__).parents[2]
_db_path = _project_root / "data" / "classiflow.db"
DB_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

print(f"DB path : {_db_path}")

engine = create_async_engine(DB_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

print("database ready")

DB path : C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db
database ready


## 3 — Node helper

Node 3 takes `text: str` (already extracted) plus the `FileReceptionResult` from Node 1.
`make_node3` wires the dependencies; `run_node3` runs a single validation.

In [3]:
from sqlalchemy.ext.asyncio import AsyncSession


def make_node3(session: AsyncSession, broadcaster: EventBroadcaster) -> ContentValidationNode:
    return ContentValidationNode(
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=broadcaster,
    )


def _pdf_reception(mime: str = "application/pdf") -> FileReceptionResult:
    return FileReceptionResult(
        passed=True,
        sha256="demo-sha256",
        detected_mime=mime,
        file_size_bytes=1024,
    )


async def run_node3(
    job_id: str,
    filename: str,
    text: str,
    mime: str = "application/pdf",
) -> object:
    async with session_factory() as session:
        result = await make_node3(session, EventBroadcaster()).run(
            job_id, filename, text, _pdf_reception(mime)
        )
        await session.commit()
    return result


print("helpers ready")

helpers ready


## 4 — OCR routing: image-only PDF

When a PDF yields no extracted text (fewer characters than `ocr_char_threshold`),
Node 3 sets `requires_ocr=True` and routes the job to the Stage 2 OCR pipeline
instead of rejecting it.

Expected: `passed=False`, `requires_ocr=True`.

In [ ]:
result = await run_node3("demo-a3-ocr", "scan.pdf", "", mime="application/pdf")

print("=== Node 3 — OCR routing ===")
print(f"  passed        : {result.passed}")
print(f"  requires_ocr  : {result.requires_ocr}")
print(f"  char_count    : {result.char_count}")
print(f"  rejection     : {result.rejection_reason}")

## 5 — Inspect the audit record

Node 3 writes an audit entry for every execution, including the OCR-routing branch.

In [ ]:
async with session_factory() as session:
    records = await SqlAuditRepository(session).list_for_job("demo-a3-ocr")

print("=== Audit records ===")
for r in records:
    print(f"  node          : {r.node}")
    print(f"  event         : {r.event}")
    print(f"  duration_ms   : {r.duration_ms} ms")
    print(f"  detail        : {r.detail}")
    print()

## 6 — Observe SSE events in real time

Node 3 emits `STARTED` then `PASSED`/`FAILED` on every execution.
Subscribe before calling the node to capture both events.

In [ ]:
SPANISH_TEXT = (
    "El Concejo Municipal de Rosario sanciona la siguiente ordenanza: "
    "Artículo 1º — Apruébase el presupuesto municipal para el ejercicio fiscal "
    "correspondiente al año en curso, conforme al detalle que se adjunta como Anexo I "
    "de la presente norma. La presente ordenanza entrará en vigencia a partir de su "
    "promulgación y publicación en el Boletín Oficial Municipal."
)

broadcaster_sse = EventBroadcaster()
events: list[object] = []


async def collect() -> None:
    async for event in broadcaster_sse.subscribe("demo-a3-sse"):
        events.append(event)
        print(f"  SSE → node={event.node}  status={event.status}")


async with session_factory() as session:
    collect_task = asyncio.create_task(collect())
    await asyncio.sleep(0)  # yield so collect() starts subscribing before run() emits

    await make_node3(session, broadcaster_sse).run(
        "demo-a3-sse", "doc.pdf", SPANISH_TEXT, _pdf_reception()
    )
    await broadcaster_sse.close("demo-a3-sse")
    await collect_task
    await session.commit()

print(f"\ncollected {len(events)} events (2 per node = 2 total)")

## 7 — Decision matrix

Each row exercises a different validation branch:

| Case | Expected outcome |
|------|------------------|
| Empty PDF | `requires_ocr=True` |
| Text below `min_chars` | `passed=False` — too short |
| Non-Spanish text | `passed=False`, `needs_agent_review=True` |
| Valid Spanish text | `passed=True` (via SLM) |

In [7]:
SHORT_TEXT = "Short text."  # 11 chars — above ocr threshold, below min_chars (50)
ENGLISH_TEXT = (
    "This document outlines the annual budget proposal for the municipal government. "
    "All expenditures must be approved by the finance committee before disbursement."
)

cases = [
    ("a3-ocr", "scan.pdf", "", "application/pdf"),
    ("a3-short", "brief.pdf", SHORT_TEXT, "application/pdf"),
    ("a3-english", "doc_en.pdf", ENGLISH_TEXT, "application/pdf"),
    ("a3-spanish", "ordenanza.pdf", SPANISH_TEXT, "application/pdf"),
]


async def _run_case(job_id: str, filename: str, text: str, mime: str) -> tuple[object, str | None]:
    try:
        result = await run_node3(job_id, filename, text, mime)
    except Exception as exc:  # noqa: BLE001
        return None, str(exc)
    else:
        return result, None


print(f"{'job':12} {'filename':16} {'passed':7} {'ocr':5} {'review':7} {'rejection'}")
print("-" * 80)

for job_id, filename, text, mime in cases:
    result, error = await _run_case(job_id, filename, text, mime)
    if error:
        print(f"{job_id:12} {filename:16} {'—':7} {'—':5} {'—':7} ERROR: {error}")
    else:
        print(
            f"{job_id:12} {filename:16} {result.passed!s:7}"
            f" {result.requires_ocr!s:5} {result.needs_agent_review!s:7}"
            f" {result.rejection_reason or '—'}"
        )

2026-08-06 00:42:27.343 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=a3-ocr node=node3_content_validation event=failed
2026-08-06 00:42:27.350 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=a3-short node=node3_content_validation event=failed


job          filename         passed  ocr   review  rejection
--------------------------------------------------------------------------------
a3-ocr       scan.pdf         False   True  False   Image-only PDF: routed to OCR pipeline
a3-short     brief.pdf        False   False False   Text too short: 11 chars (min 50)


2026-08-06 00:42:33.228 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=a3-english node=node3_content_validation event=passed


a3-english   doc_en.pdf       True    False False   —


2026-08-06 00:42:58.979 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=a3-spanish node=node3_content_validation event=failed


a3-spanish   ordenanza.pdf    False   False True    SLM: The text appears to be a formal document related to municipal budget approval.


## 8 — Run with real text files from disk

Drop any `.txt` file with extracted document text into:
```
src/classiflow/playground/samples/
```
then run the cell — it processes every `.txt` file through Node 3.

In [8]:
from pathlib import Path

import IPython.display as ipyd
from IPython.display import HTML

samples_dir = next(
    p / "samples"
    for p in [Path.cwd(), Path.cwd() / "src" / "classiflow" / "playground"]
    if (p / "samples").is_dir()
)
txt_files = sorted(samples_dir.glob("*.txt"))
if not txt_files:
    msg = f"No .txt files in {samples_dir}. Drop a text file there first."
    raise FileNotFoundError(msg)


async def _run_real_file(job_id: str, file_path: Path) -> tuple[object, object, str | None]:
    text = file_path.read_text(encoding="utf-8", errors="replace")  # noqa: ASYNC240
    try:
        async with session_factory() as session:
            repo = SqlAuditRepository(session)
            result = await make_node3(session, EventBroadcaster()).run(
                job_id, file_path.name, text, _pdf_reception()
            )
            a3_rec = next(
                (
                    rec
                    for rec in await repo.list_for_job(job_id)
                    if rec.node == "node3_content_validation"
                ),
                None,
            )
            await session.commit()
    except Exception as exc:  # noqa: BLE001
        return None, None, str(exc)
    else:
        return result, a3_rec, None


for file_path in txt_files:
    job_id = f"demo-real-{file_path.stem}"
    result, a3_record, error_msg = await _run_real_file(job_id, file_path)

    if error_msg:
        status_icon, status_color = "⚠️ ERROR", "#e65100"
        rows = [("File", file_path.name), ("Error", error_msg)]
    else:
        status_icon = "✅ PASSED" if result.passed else "❌ FAILED"
        status_color = "#2e7d32" if result.passed else "#c62828"
        rows = [
            ("File", file_path.name),
            ("Chars", str(result.char_count)),
            ("Language", result.detected_language or "—"),
            ("Requires OCR", str(result.requires_ocr)),
            ("Needs review", str(result.needs_agent_review)),
            ("Duration (node3)", f"{a3_record.duration_ms} ms" if a3_record else "—"),
            ("Rejection reason", result.rejection_reason or "—"),
        ]

    rows_html = "".join(
        f'<tr><td style="color:#555;padding:4px 12px 4px 0;white-space:nowrap">'
        f"{k}</td>"
        f'<td style="font-family:monospace;padding:4px 0">{v}</td></tr>'
        for k, v in rows
    )
    ipyd.display(
        HTML(f"""
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;
                    margin:8px 0;font-family:sans-serif;max-width:540px">
          <div style="font-size:1.1em;font-weight:bold;color:{status_color};
                      margin-bottom:10px">{status_icon}</div>
          <table style="border-collapse:collapse;width:100%">{rows_html}</table>
        </div>
        """)
    )

2026-08-06 00:43:25.870 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-real-test node=node3_content_validation event=passed


File,test.txt
Chars,1110021
Language,es
Requires OCR,False
Needs review,False
Duration (node3),12953 ms
Rejection reason,—


## 9 — Cleanup

In [9]:
await engine.dispose()
print("engine disposed — database file is now unlocked")

engine disposed — database file is now unlocked
